In [1]:
%pip install langchain
%pip install ctransformers
%pip install sentence-transformers
%pip install pinecone-client
%pip install pypdf
%pip install python_dotenv
%pip install langchain-pinecone
%pip install langchain-community
%pip install langchain-core
%pip install langchain-huggingface
%pip install langchain-classic
%pip install pinecone
%pip install numpy
%pip install transformers
%pip install huggingface-hub
%pip install scikit-learn
%pip install scipy
%pip install tqdm
%pip install py-cpuinfo>=9.0.0,<10.0.0
# For CPU-only PyTorch wheel (recommended on machines without CUDA):
%pip install --index-url https://download.pytorch.org/whl/cpu/ torch

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you 

In [2]:
try:
    import traceback
    import langchain_classic
    from langchain_classic.chains import RetrievalQA
    from langchain_pinecone import PineconeVectorStore as langchainpinecone
    import pinecone
    import numpy as np
    print('IMPORT-TEST: all imports OK')
except Exception:
    traceback.print_exc()

/media/windows_disk/ML/AI_ML_Code/ALLinOne/PDF-ChatbotUsing_RAG/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


IMPORT-TEST: all imports OK


In [3]:
from langchain_classic.chains import RetrievalQA 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore as langchainpinecone
from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import CTransformers
from pinecone import Pinecone
from dotenv import load_dotenv
import os

In [4]:
def load_pdf(data):
    loader=DirectoryLoader(data,
                           glob="*.pdf",
                           loader_cls=PyPDFLoader)
    documents=loader.load()
    return documents

In [5]:
base_path = os.getcwd()
one_level_up = os.path.dirname(base_path)
full_path = os.path.join(one_level_up,"data")
print(full_path)

/media/windows_disk/ML/AI_ML_Code/ALLinOne/PDF-ChatbotUsing_RAG/data


In [6]:
extracted_data=load_pdf(full_path)

In [7]:
def text_spit(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [8]:
text_chunks=text_spit(extracted_data)
print("length of my chunk :",len(text_chunks))

length of my chunk : 5859


In [9]:
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [10]:
embeddings=download_hugging_face_embeddings()

In [11]:
try:
    emb = download_hugging_face_embeddings()
    print('Embeddings object:', type(emb))
    v = emb.embed_query('Hello, world')
    print('Embed test length:', len(v))
except Exception:
    import traceback; traceback.print_exc()

Embeddings object: <class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
Embed test length: 384


In [12]:
load_dotenv()

True

In [13]:
PINECONE_API_KEY =os.environ.get("PINECONE_API_KEY")
PINECONE_INDEX = os.environ.get("PINECONE_INDEX")

## Enter inder_name below

In [14]:
pc=Pinecone(api_key=PINECONE_API_KEY)

In [15]:
index_name=PINECONE_INDEX
docsearch=langchainpinecone.from_texts([t.page_content for t in text_chunks],embedding=embeddings,index_name=index_name)

In [16]:
prompt_templete="""
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

In [17]:
PROMPT=PromptTemplate(template=prompt_templete,input_variables=["context","question"])
chain_type_kwargs={"prompt":PROMPT}

In [22]:
model_path = os.path.join(one_level_up,"project/model/llama-2-7b-chat.ggmlv3.q4_0.bin")

# Guard: check model file exists before attempting to load
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found at {model_path}. Please download/copy the ggml file into the 'model/' directory or update `model_path`.")

llm=CTransformers(model=model_path,
                  model_type="llama",
                  config={"max_new_tokens" :800,
                          "temperature" : 0.8})

## Model file check
Make sure the ggml model file is placed at `model/llama-2-7b-chat.ggmlv3.q4_0.bin`.
If you don't have it, download or copy it into the `model/` folder or update `model_path` in the cell before running.

In [19]:
qa=RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(search_kwargs={"k":2}),
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs)

In [20]:
while True:
    user_input=input(f"Input Prompt : ")
    if (user_input=="stop"):
        break
    result=qa.invoke({"query":user_input})
    print("Response : ",result["result"])

Response :  Antiprotozoal drugs are used to treat infections caused by protozoa because these microorganisms have developed resistance to many other types of antibiotics. Protozoa have a unique cell structure that makes them difficult to target with traditional antibiotics, so antiprotozoal drugs were developed specifically to combat these infections.


In [21]:
# === Pinecone: simple clear data ===
# Enter an index name and this will delete ALL vectors from that index (keeps index itself).
# WARNING: This action is irreversible. Run only if you are sure.
from pinecone import Pinecone
import os

pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
indexes = pc.list_indexes()
print("Available indexes:", indexes)

index_to_use = input("Enter index name to delete all vectors (leave blank to cancel): ").strip()
if not index_to_use:
    print("No index provided — aborting.")
else:
    try:
        idx = pc.Index(index_to_use)
    except Exception as e:
        print(f"Error: could not open index '{index_to_use}':", e)
    else:
        try:
            print(f"Deleting all vectors from index '{index_to_use}'...")
            idx.delete(delete_all=True)
            print("All vectors deleted.")
        except Exception as e:
            print("Error during deletion:", e)

Available indexes: [{
    "name": "medical-chatbot",
    "metric": "cosine",
    "host": "medical-chatbot-xbcxr12.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "embed": {
        "model": "llama-text-embed-v2",
        "field_map": {
            "text": "text"
        },
        "dimension": 384,
        "metric": "cosine",
        "write_parameters": {
            "dimension": 384.0,
            "input_type": "passage",
            "truncate": "END"
        },
        "read_parameters": {
            "dimension": 384.0,
            "input_type": "query",
            "truncate": "END"
        },
        "vector_type": "dense"
    }
}]
Deleting all vectors from index 'medical-chatbot'...
All vectors delete